# R7.4｜R6.5-CrossDrop60 冻结权重复算 → ADT 逐目标日期独立验证

**单一 Notebook · WSL/VS Code · RTX 5080 · 无训练 · 2026-10-16**

本项目忠实复用用户此前 R6.5 Notebook 的模型定义、180 日窗口、训练期 EOF、早停及模型合同签名，从 **`best.pt`** 读取冻结权重。

1. 仅导出已训练的 **2014/2019 × seed 71/72 × lead 30/60/90 天** 的真实预测；缺 checkpoint 就明确失败，不用训练或替代模型。
2. 与独立卫星 ADT 流轴 **同一目标日期** 配对；主要指标为主流区 Branch2 在 122.0°E—144.3°E 的对称路径距离，Branch1 单独报告；绝不把两个分支用人为直线连起来。
3. 计算 A/B/PN/C/D/E 六断面交点误差、月尺度变化和配对覆盖率。逐日重叠起报与种子不视为独立观测。
4. 自动输出英语论文图、诊断表、抽样真实预测轨迹、ZIP 和 SHA256；结果仅属**研发年份的独立参考产品比较**，不能宣称为未使用年份的盲测。

**重要：** ADT 是独立数据产品，不等于没有误差的真值；卫星表层锋面和 GLORYS 50 m 速度核心有物理差异。R7.3 的 2023 部分轨迹 QC 不用于本轮 2014/2019 预测主表。

In [ ]:
# 0 配置，全部路径为 WSL 格式。无训练，不访问外网。
from pathlib import Path
from dataclasses import dataclass,asdict
from datetime import datetime,timezone
import os,re,math,json,hashlib,zipfile,time,traceback,warnings,random,gc
import numpy as np,pandas as pd
from scipy.spatial import cKDTree
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
import matplotlib.pyplot as plt
from IPython.display import display,FileLink
from tqdm.auto import tqdm

@dataclass
class Settings:
    ROOT:str='/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL:str='final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    ADT_AXES_REL:str='R70_ADT_Independent_Python/adt_axes_npz'
    WEIGHTS_REL:str='R6_5_跨分支交换随机屏蔽实验/模型权重/R65_Checkpoints'
    OUTPUT_REL:str='R7_4_R65_CrossDrop60_ADT_ForecastValidation'
    HISTORY:int=180
    LEADS:tuple=(30,60,90)
    QA_YEARS:tuple=(2014,2019)
    SEEDS:tuple=(71,72)
    PURGE_DAYS:int=7
    MAX_INPUT_GAP:int=2
    MAX_MISSING_FRAC:float=.03
    DROP_BRANCH1_FIRST:int=12
    BRANCH2_MAX_LON:float=145.
    EOF_VARIANCE_TARGET:float=.99
    BATCH_SIZE:int=32
    DROPOUT:float=.30
    LR:float=.001
    WEIGHT_DECAY:float=.0001
    DENSIFY_KM:float=5.
    TERMINAL_LON:float=144.3
    MIN_ADT_DAYS_FOR_YEAR:int=10
    BOOT_REPS:int=400
    FIG_DPI:int=240
    SAVE_PER_SAMPLE_NPZ:bool=True
    INCLUDE_ALL_PREDICTIONS_IN_ZIP:bool=False
    ZIP_SAMPLE_PREDICTIONS:int=24
    EXPORT_ONLY:bool=False
    FAIL_IF_INCOMPLETE:bool=True

c=Settings();ROOT=Path(c.ROOT);DS=ROOT/c.DATASET_REL;ADTDIR=ROOT/c.ADT_AXES_REL; WEIGHTS=ROOT/c.WEIGHTS_REL
RUN=ROOT/c.OUTPUT_REL;PATHS=RUN/'forecast_paths';TABLES=RUN/'tables'; FIG=RUN/'figures_en'; REP=RUN/'reports';SNAP=RUN/'preprocessing_snapshots'
for q in (RUN,PATHS,TABLES,FIG,REP,SNAP):q.mkdir(parents=True,exist_ok=True)
ZIP=ROOT/'R74_上传给ChatGPT_R65单模型预测vsADT独立验证.zip'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device',DEV,'Torch',torch.__version__)
print(json.dumps(asdict(c),ensure_ascii=False,indent=2))
for label,p in [('frozen dataset',DS),('R65 best weights',WEIGHTS),('ADT full axes',ADTDIR)]:
 print(label, p, 'exists=',p.exists())
 if not p.exists():raise FileNotFoundError(f'缺少{label}: {p}，不可重训或推断替代')

def file_sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
 return h.hexdigest()
DATASET_SHA=file_sha256(DS)

In [ ]:
# 1 按 R6.5 原始程序重建 11,771天冻结表示 / 因果填补 / anchors。
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
if not(len(dates)==len(b1)==len(b2)):raise RuntimeError('时间与两条分支样本不一致')
if dates.has_duplicates:raise RuntimeError('重复日期')
o=np.argsort(dates.values);dates=dates[o];b1=b1[o];b2=b2[o]
if not(np.isfinite(b1).all() and np.isfinite(b2).all()):raise RuntimeError('原始路径含NaN/Inf')
b1=b1[:,c.DROP_BRANCH1_FIRST:];lat1=lat1[c.DROP_BRANCH1_FIRST:]
mask2=lon2<=c.BRANCH2_MAX_LON+1e-8;b2=b2[:,mask2];lon2=lon2[mask2]
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
if (pos<0).any():raise RuntimeError('日历索引错误')
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1;A2[pos]=b2
F1=pd.DataFrame(A1).ffill(limit=c.MAX_INPUT_GAP).to_numpy(np.float32)
F2=pd.DataFrame(A2).ffill(limit=c.MAX_INPUT_GAP).to_numpy(np.float32)
H=int(c.HISTORY);L=max(c.LEADS)
missing=(~valid).astype(np.int32);idx=[]
for i in range(H-1,len(cal)-L):
 if not valid[i]:continue
 win=missing[i-H+1:i+1]
 if win.mean()>c.MAX_MISSING_FRAC:continue
 if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>c.MAX_INPUT_GAP:continue
 if any(not valid[i+d] for d in c.LEADS):continue
 if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()):continue
 idx.append(i)
ISS=np.asarray(idx,np.int32)
if not len(ISS):raise RuntimeError('找不到180/30/60/90日原始起报anchors')
assert all(np.all((cal[ISS+lead]-cal[ISS]).days==lead) for lead in c.LEADS)
print('Dataset signature:',DATASET_SHA,'anchors=',len(ISS),'dims',b1.shape[1],b2.shape[1])

In [ ]:
# 2 训练期 EOF 与原版模型原语：以下核心类直接从 R6.5 正式 Notebook 提取。
class R65Cfg:
    DROPOUT=0.30
cfg=R65Cfg()
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal < f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],c.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],c.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained < c.EOF_VARIANCE_TARGET-1e-6:raise RuntimeError('EOF解释方差不达99%')
    z1=pp1.z(F1);z2=pp2.z(F2)
    p1=pp1.pc(F1);p2=pp2.pc(F2)
    np.savez_compressed(SNAP/f"QA_{f['year']-1}_trainonly_EOF.npz",
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,
        b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        lat1=lat1,lon2=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return dict(pp1=pp1,pp2=pp2,z1=z1,z2=z2,p1=p1,p2=p2)

def qa_split(year):
    qa_start=pd.Timestamp(year,1,1);next_start=pd.Timestamp(year+1,1,1)
    inner_start=pd.Timestamp(year-1,1,1);purge=pd.Timedelta(days=c.PURGE_DAYS)
    train=ISS[cal[ISS+L] < inner_start-purge]
    val=ISS[(cal[ISS]>=inner_start) & (cal[ISS+L]<qa_start-purge)]
    qa=ISS[(cal[ISS]>=qa_start) & (cal[ISS+L]<next_start-purge)]
    if min(len(train),len(val),len(qa))<100 or len(train)<500:
        raise RuntimeError(f'QA {year} train={len(train)} val={len(val)} qa={len(qa)}')
    assert cal[train+L].max()<cal[val].min()-purge
    assert cal[val+L].max()<cal[qa].min()-purge
    return {'fold':year,'year':year+1,'train':train,'val':val,'qa':qa,'train_cut':inner_start-purge}
FOLDS={int(year):qa_split(int(year)) for year in c.QA_YEARS}
print('R65 fold audit',[(y,len(f['train']),len(f['val']),len(f['qa'])) for y,f in FOLDS.items()])

### 从 R6.5 原始 Notebook 逐类移植模型（保留 state_dict 键名不变）

In [ ]:
# 3 原架构代码：不更改 GroupNorm 分组数、Dropout、输入特征或层名。
import torch.nn.functional as F
class ChannelSeparatedEncoder(nn.Module):
    """单时刻空间卷积；输入[B,T,2*M] => Conv1d [B*T,2,M]."""
    def __init__(self,feature,branch):
        super().__init__()
        assert feature%2==0, '需要[x, delta]等长拼接输入'
        self.raw_dim=feature//2
        base_c=16 if branch==1 else 32
        k=3 if branch==1 else 5
        channels=base_c*2
        # 正确的对称空间填充：dilation=2 -> padding=k-1，输出仍为M
        self.conv=nn.Conv1d(2,channels,kernel_size=k,dilation=2,padding=k-1)
        self.bn=nn.BatchNorm1d(channels)
        self.fc=nn.Linear(channels*self.raw_dim,32 if branch==1 else 64)
        self.out_dim=self.fc.out_features
        self.drop=nn.Dropout(cfg.DROPOUT)
    def channelize(self,x):
        B,T,F=x.shape
        if F!=self.raw_dim*2: raise RuntimeError('R53特征维不匹配')
        return torch.stack((x[:,:,:self.raw_dim],x[:,:,self.raw_dim:]),dim=2).reshape(B*T,2,self.raw_dim)
    def forward(self,x):
        B,T,F=x.shape
        z=self.channelize(x)
        z=self.drop(torch.relu(self.bn(self.conv(z))))
        if z.shape[2]!=self.raw_dim:raise RuntimeError('R53 spatial-conv输出宽度发生变化')
        z=self.drop(torch.relu(self.fc(z.flatten(1))))
        return z.reshape(B,T,-1)
class ChannelHybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,projection=None,use_pc_skip=False):
        super().__init__()
        self.enc=ChannelSeparatedEncoder(feature,branch)
        self.use_pc_skip=bool(use_pc_skip)
        self.raw_dim=feature//2
        self.pc_dim=pc_dim
        if self.use_pc_skip:
            self.register_buffer('V',torch.as_tensor(projection,dtype=torch.float32))
        hidden=32 if branch==1 else 64
        in_dim=self.enc.out_dim+(2*pc_dim if self.use_pc_skip else 0)
        self.rnn=nn.LSTM(in_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
    def eof_skip(self,x):
        if not self.use_pc_skip: raise RuntimeError('no PC skip in this model')
        return torch.cat((x[:,:,:self.raw_dim]@self.V,x[:,:,self.raw_dim:]@self.V),dim=-1)
    def forward(self,x):
        z=self.enc(x)
        if self.use_pc_skip: z=torch.cat((z,self.eof_skip(x)),dim=-1)
        h,_=self.rnn(z)
        return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def make_r53_model(name,branch,fd):
    if name!='R53-ChannelConv-PCConcat':raise ValueError('只允许原R53-PCConcat编码器')
    feature=fd[f'z{branch}'].shape[1]*2; pdim=fd[f'pp{branch}'].n
    return ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
class JointHiddenEOF(nn.Module):
    def __init__(self,fd,mode='cross'):
        super().__init__()
        if mode not in ('cross','self'): raise ValueError(mode)
        self.mode=mode
        self.net1=make_r53_model('R53-ChannelConv-PCConcat',1,fd)
        self.net2=make_r53_model('R53-ChannelConv-PCConcat',2,fd)
        d1=int(self.net1.fc.in_features);d2=int(self.net2.fc.in_features)
        self.exchange1=nn.Linear(d2 if mode=='cross' else d1,d1)
        self.exchange2=nn.Linear(d1 if mode=='cross' else d2,d2)
        nn.init.zeros_(self.exchange1.weight);nn.init.zeros_(self.exchange1.bias)
        nn.init.zeros_(self.exchange2.weight);nn.init.zeros_(self.exchange2.bias)
    def _hidden(self,net,x):
        a=net.enc(x)
        a=torch.cat([a,net.eof_skip(x)],dim=-1)
        h,_=net.rnn(a)
        return h[:,-1]
    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        if self.mode=='cross':
            t1=h1+self.exchange1(torch.tanh(h2))
            t2=h2+self.exchange2(torch.tanh(h1))
        else:
            t1=h1+self.exchange1(torch.tanh(h1))
            t2=h2+self.exchange2(torch.tanh(h2))
        o1=self.net1.fc(t1).reshape(len(x1),3,self.net1.pc_dim)
        o2=self.net2.fc(t2).reshape(len(x2),3,self.net2.pc_dim)
        return o1,o2
class CrossExchangeDropGN(JointHiddenEOF):
    def __init__(self,fd,drop_p):
        super().__init__(fd,'cross')
        self.drop_p=float(drop_p)
        assert 0 <= self.drop_p < 1.0
        for net in (self.net1,self.net2):
            ch=net.enc.conv.out_channels
            assert ch%8==0
            net.enc.bn=nn.GroupNorm(8,ch)

    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        m1=self.exchange1(torch.tanh(h2))
        m2=self.exchange2(torch.tanh(h1))
        # 训练期对隐藏特征做独立 dropout；推理时恒等，严禁随机测试增强
        m1=F.dropout(m1,p=self.drop_p,training=self.training)
        m2=F.dropout(m2,p=self.drop_p,training=self.training)
        y1=self.net1.fc(h1+m1).reshape(len(x1),3,self.net1.pc_dim)
        y2=self.net2.fc(h2+m2).reshape(len(x2),3,self.net2.pc_dim)
        return y1,y2

def r65_model(fd,name):
    if name!='R65-CrossDrop60':raise ValueError('只能读取冻结R65-CrossDrop60')
    return CrossExchangeDropGN(fd,0.60)

In [ ]:
# 4 精确重构原 R6.5 cid 合同。CID 不吻合则不加载，不降级。
R61_ALPHA=.40
R65_SEEDS=(71,72)
R65_EXPLORATORY_YEARS=(2014,2019)
R65_CONFIRM_YEARS=(2006,2011)
R65_EPOCHS=36
R65_PATIENCE=8
R65_PROTOCOL={
    'version':'R65_GN8_CrossExchangeDrop_v1',
    'dataset_sha256':DATASET_SHA,
    'data_representation':'Original frozen',
    'history_days':c.HISTORY,
    'lead_days':list(c.LEADS),
    'physical_loss_alpha':float(R61_ALPHA),
    'development_years':list(R65_EXPLORATORY_YEARS),
    'confirmation_development_years':list(R65_CONFIRM_YEARS),
    'seeds':list(R65_SEEDS),
    'epochs':R65_EPOCHS,'patience':R65_PATIENCE,
    'dropout':c.DROPOUT,
    'cross_exchange_dropout':[.35,.60],
    'lr':c.LR,'weight_decay':c.WEIGHT_DECAY,
    'reference':'same-data same-split R6.4 genuine prediction archive',
    'model_choice_from_confirm_years':False,
    'all_QA_years_development_not_blind':True
}
def r65_contract(f,fd,seed,name):
    c={**R65_PROTOCOL,'QA_year':int(f['year']-1),'seed':int(seed),'name':name,
       'tr_end':int(f['train'][-1]),'va_end':int(f['val'][-1]),
       'nPC1':int(fd['pp1'].n),'nPC2':int(fd['pp2'].n)}
    return hashlib.sha256(json.dumps(c,ensure_ascii=False,sort_keys=True).encode()).hexdigest()

class JointPairDataset(Dataset):
    def __init__(self,issues,fd):
        self.issues=np.asarray(issues,dtype=np.int64)
        self.z1=fd['z1'];self.z2=fd['z2'];self.p1=fd['p1'];self.p2=fd['p2']
    def __len__(self):return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j])
        def pack(z):
            v=z[i-H+1:i+1]
            d=np.empty_like(v);d[0]=0;d[1:]=v[1:]-v[:-1]
            return torch.from_numpy(np.concatenate([v,d],axis=-1).copy())
        yy1=np.stack([self.p1[i+lead] for lead in c.LEADS]);yy2=np.stack([self.p2[i+lead] for lead in c.LEADS])
        return pack(self.z1),pack(self.z2),torch.from_numpy(yy1.copy()),torch.from_numpy(yy2.copy())

def r65_predict_pair(model,fd,issues):
    ld=DataLoader(JointPairDataset(issues,fd),batch_size=c.BATCH_SIZE,shuffle=False,num_workers=0)
    all1=[];all2=[];model.eval()
    with torch.inference_mode():
        for x1,x2,*_ in ld:
            p1,p2=model(x1.to(DEV),x2.to(DEV))
            all1.append(p1.float().cpu().numpy());all2.append(p2.float().cpu().numpy())
    return fd['pp1'].invert(np.concatenate(all1)),fd['pp2'].invert(np.concatenate(all2))

checkrows=[]
for year,f in FOLDS.items():
    fd=make_fold_data(f)
    net=r65_model(fd,'R65-CrossDrop60')
    for seed in c.SEEDS:
        pt=WEIGHTS/str(year)/str(seed)/'R65-CrossDrop60'/'best.pt'
        expected=r65_contract(f,fd,seed,'R65-CrossDrop60')
        row={'qa_year':year,'seed':seed,'best_pt':str(pt),'file_exists':pt.is_file(),
             'expected_cid':expected,'passed':False,'reason':''}
        if pt.is_file():
            try:
                ck=torch.load(pt,map_location='cpu',weights_only=True)
                if ck.get('cid')!=expected:raise RuntimeError('CID mismatch')
                net.load_state_dict(ck['state'],strict=True)
                row['passed']=True;row['checkpoint_sha256']=file_sha256(pt)
                row['checkpoint_epoch']=ck.get('epoch')
            except Exception as e:row['reason']=repr(e)
        else:row['reason']='missing frozen best.pt'
        checkrows.append(row)
    del fd,net
ckpt_audit=pd.DataFrame(checkrows)
ckpt_audit.to_csv(TABLES/'00_冻结权重合同审计.csv',index=False,encoding='utf-8-sig')
display(ckpt_audit[['qa_year','seed','file_exists','passed','reason']])
if not ckpt_audit['passed'].all() and c.FAIL_IF_INCOMPLETE:
    print('缺失/失配组合不可凭已有CSV重造路径。请将R6.5本地best.pt恢复到指定路径或调整WEIGHTS_REL。')
    # Don't raise until after lightweight archive step; real export will check this again.

In [ ]:
# 5 只使用通过 CID 的 best.pt 导出 30/60/90天真实预测，经纬度文件与索引完整落盘。
manifest_path=TABLES/'01_真实预测路径Manifest.csv'
old=(pd.read_csv(manifest_path) if manifest_path.is_file() else pd.DataFrame())
rows=[];status=[]
for year,f in FOLDS.items():
    fd=make_fold_data(f)
    for seed in c.SEEDS:
        audit=ckpt_audit[(ckpt_audit.qa_year==year)&(ckpt_audit.seed==seed)].iloc[0]
        if not audit['passed']:
            status.append({'qa_year':year,'seed':seed,'status':'no_verified_checkpoint','detail':audit.reason})
            continue
        pt=Path(audit.best_pt)
        # 缺少任何一个预测轨迹时整组使用已签名权重重推理，无目标真值参与。
        issue_list=np.asarray(f['qa']);expected_files=[]
        for issue in issue_list:
            for lead in c.LEADS:
                stem=f'R65-CrossDrop60_{year}_s{seed}_{cal[issue]:%Y-%m-%d}_L{lead}'
                expected_files.append(PATHS/(stem+'.npz'))
        need_predict=(not c.SAVE_PER_SAMPLE_NPZ or any(not p.exists() for p in expected_files))
        if not need_predict:
            # 仅在每一个文件的权重签名仍完全一致时才允许复用。
            current_sha=file_sha256(pt)
            for path in expected_files:
                try:
                    with np.load(path,allow_pickle=False) as existing:
                        if str(existing['checkpoint_sha256'])!=current_sha:
                            need_predict=True;break
                except Exception:
                    need_predict=True;break
        if need_predict:
            net=r65_model(fd,'R65-CrossDrop60').to(DEV)
            ck=torch.load(pt,map_location='cpu',weights_only=True)
            if ck.get('cid')!=r65_contract(f,fd,seed,'R65-CrossDrop60'):
                raise RuntimeError('CID mismatch during inference')
            net.load_state_dict(ck['state'],strict=True)
            net.eval();t0=time.perf_counter()
            p1,p2=r65_predict_pair(net,fd,issue_list)
            infertime=time.perf_counter()-t0
            if p1.shape!=(len(issue_list),3,len(lat1)) or p2.shape!=(len(issue_list),3,len(lon2)):
                raise RuntimeError(f'预测形状不匹配：{p1.shape} {p2.shape}')
            groupfile=PATHS/f'R65_{year}_s{seed}_all_issues.npz'
            np.savez_compressed(groupfile,p1=p1.astype(np.float32),p2=p2.astype(np.float32),
                issue_date=np.asarray([str(cal[i].date()) for i in issue_list]),
                target_dates=np.asarray([[str(cal[i+d].date()) for d in c.LEADS] for i in issue_list]),
                leads=np.asarray(c.LEADS),dataset_sha256=DATASET_SHA,
                checkpoint_sha256=file_sha256(pt))
        else:infertime=np.nan
        for issue in issue_list:
            for j,lead in enumerate(c.LEADS):
                issue_date=str(cal[issue].date());target_date=str(cal[issue+lead].date())
                assert (pd.Timestamp(target_date)-pd.Timestamp(issue_date)).days==lead
                path=PATHS/f'R65-CrossDrop60_{year}_s{seed}_{issue_date}_L{lead}.npz'
                rows.append({'model':'R65-CrossDrop60','seed':seed,'qa_year':year,
                    'issue_date':issue_date,'target_date':target_date,'lead_days':lead,
                    'path_file':str(path),'checkpoint_sha256':file_sha256(pt),'dataset_sha256':DATASET_SHA})
        if need_predict and c.SAVE_PER_SAMPLE_NPZ:
            for i,issue in enumerate(issue_list):
                for j,lead in enumerate(c.LEADS):
                    p=PATHS/f'R65-CrossDrop60_{year}_s{seed}_{cal[issue]:%Y-%m-%d}_L{lead}.npz'
                    np.savez_compressed(p,branch1_lon=p1[i,j].astype(np.float32),branch1_lat=lat1.astype(np.float32),
                        branch2_lon=lon2.astype(np.float32),branch2_lat=p2[i,j].astype(np.float32),
                        issue_date=str(cal[issue].date()),target_date=str(cal[issue+lead].date()),
                        lead_days=lead,seed=seed,qa_year=year,checkpoint_sha256=file_sha256(pt))
        status.append({'qa_year':year,'seed':seed,'status':'exported' if need_predict else 'reused_existing',
                       'n_issues':len(issue_list),'n_lead_predictions':len(issue_list)*len(c.LEADS),
                       'inference_wall_s':infertime,'checkpoint_sha256':file_sha256(pt)})
    del fd
    gc.collect()
    if DEV.type=='cuda':torch.cuda.empty_cache()
manifest=pd.DataFrame(rows).drop_duplicates(['qa_year','seed','issue_date','lead_days']) if rows else pd.DataFrame(columns=['model','seed','qa_year','issue_date','target_date','lead_days','path_file','checkpoint_sha256','dataset_sha256'])
manifest.to_csv(manifest_path,index=False,encoding='utf-8-sig')
pd.DataFrame(status).to_csv(TABLES/'02_冻结模型推理导出审计.csv',index=False,encoding='utf-8-sig')
print('Genuine forecast files:',len(manifest),'checkpoints',len(status),'vs required',len(FOLDS)*len(c.SEEDS))
if not len(manifest):print('NO FORECAST OUTPUT: missing checkpoint(s); do not interpret as 0 error')

In [ ]:
# 6 纯几何对照：球面距离 + 共域约束 + 六断面线段交点，不使用预测真值改变轨迹。
EARTH_KM=6371.0088
SECTIONS={
 'A':((121.50,23.50),(125.00,22.75)),
 'B':((121.50,26.25),(124.25,24.25)),
 'PN':((126.00,29.00),(128.15,27.30)),
 'C':((130.25,31.25),(132.50,27.00)),
 'D':((137.00,34.75),(138.00,27.75)),
 'E':((141.75,40.00),(145.00,31.50)),
}

def xyz(x):
 a=np.deg2rad(np.asarray(x,float));lo=a[:,0];la=a[:,1]
 return EARTH_KM*np.column_stack([np.cos(la)*np.cos(lo),np.cos(la)*np.sin(lo),np.sin(la)])
def great_circle(chord):return 2*EARTH_KM*np.arcsin(np.clip(np.asarray(chord)/(2*EARTH_KM),0,1))
def densify(x,spacing=None):
 x=np.asarray(x,float);spacing=c.DENSIFY_KM if spacing is None else spacing
 if len(x)<2 or not np.isfinite(x).all():raise ValueError('Nonfinite/short axis')
 d=great_circle(np.linalg.norm(np.diff(xyz(x),axis=0),axis=1))
 if (d>250).any():raise ValueError('jump >250km; QC reject')
 out=[x[0]]
 for a,b,dist in zip(x[:-1],x[1:],d):
    n=max(1,int(np.ceil(dist/spacing)))
    out.extend(a+(b-a)*(j/n) for j in range(1,n+1))
 return np.asarray(out)
def poly_score(pred,ref):
 a=densify(pred);b=densify(ref)
 if min(len(a),len(b))<3:raise ValueError('short axis')
 u=xyz(a);v=xyz(b)
 d1=great_circle(cKDTree(v).query(u)[0]);d2=great_circle(cKDTree(u).query(v)[0]);all_dist=np.r_[d1,d2]
 return {'rmse_km':float(np.sqrt(np.mean(all_dist**2))),'mae_km':float(np.mean(all_dist)),
         'p90_km':float(np.quantile(all_dist,.90)),'directed_pred_to_adt_km':float(np.mean(d1)),
         'directed_adt_to_pred_km':float(np.mean(d2))}
def crossing(x,a,level):
 vals=np.asarray(x)[:,a];ids=np.flatnonzero((vals[:-1]<level)&(vals[1:]>=level))
 return ids
def east_prefix(x,limit=c.TERMINAL_LON):
 x=np.asarray(x,float)
 if x[0,0]>=limit:raise ValueError('already east of terminal')
 ids=crossing(x,0,limit)
 if not len(ids):raise ValueError('terminal_not_reached')
 i=int(ids[0]);t=(limit-x[i,0])/(x[i+1,0]-x[i,0])
 return np.vstack([x[:i+1],x[i]+t*(x[i+1]-x[i])]),len(ids)
def after_lon(x,west):
 x=np.asarray(x,float);ids=crossing(x,0,west)
 if not len(ids):
    if x[0,0]>=west:return x,0
    raise ValueError('west_bound_not_reached')
 i=int(ids[0]);t=(west-x[i,0])/(x[i+1,0]-x[i,0]);return np.vstack([x[i]+t*(x[i+1]-x[i]),x[i+1:]]),len(ids)
def latitude_segment(x,south,north):
 x=np.asarray(x,float)
 ids=crossing(x,1,south)
 if not len(ids):raise ValueError('south_lat_not_crossed')
 i=int(ids[0]);t=(south-x[i,1])/(x[i+1,1]-x[i,1]);x=np.vstack([x[i]+t*(x[i+1]-x[i]),x[i+1:]])
 ids=crossing(x,1,north)
 if not len(ids):raise ValueError('north_lat_not_crossed')
 i=int(ids[0]);t=(north-x[i,1])/(x[i+1,1]-x[i,1]);return np.vstack([x[:i+1],x[i]+t*(x[i+1]-x[i])]),len(ids)
def det2(a,b):return float(a[0]*b[1]-a[1]*b[0])
def section_intersections(axis,segment):
 axis=np.asarray(axis,float);ss=np.asarray(segment,float);latmean=float(ss[:,1].mean())
 scale=np.asarray([111.195*np.cos(np.deg2rad(latmean)),111.195]);q0=ss[0]*scale;r=(ss[1]-ss[0])*scale
 out=[]
 for i in range(len(axis)-1):
    a=axis[i]*scale;s=(axis[i+1]-axis[i])*scale;den=det2(s,r)
    if abs(den)<1e-10:continue
    d=q0-a;t=det2(d,r)/den;u=det2(d,s)/den
    if -1e-9<=t<=1+1e-9 and -1e-9<=u<=1+1e-9:
        lonlat=axis[i]+np.clip(t,0,1)*(axis[i+1]-axis[i])
        dis=great_circle(np.linalg.norm(xyz([lonlat])[0]-xyz([ss[0]])[0]))
        if not any(abs(dis-v['section_distance_km'])<.05 for v in out):
            out.append({'lon':float(lonlat[0]),'lat':float(lonlat[1]),
                        'section_distance_km':float(dis),'along_track_index':i+float(t)})
 return sorted(out,key=lambda d:d['along_track_index'])

def load_adt(date):
 p=ADTDIR/(pd.Timestamp(date).strftime('%Y%m%d')+'.npz')
 if not p.is_file():return None
 with np.load(p,allow_pickle=False) as z:
    a=np.column_stack([z['axis_lon'].ravel(),z['axis_lat'].ravel()]).astype(float)
 if len(a)<20 or not np.isfinite(a).all():return None
 return a

def load_prediction(row):
 with np.load(row['path_file'],allow_pickle=False) as z:
    if str(z['issue_date'])!=str(row['issue_date']) or str(z['target_date'])!=str(row['target_date']):
        raise RuntimeError('prediction file metadata inconsistent')
    if str(z['checkpoint_sha256'])!=str(row['checkpoint_sha256']):raise RuntimeError('checkpoint sha256 mismatch with saved forecast')
    if int(z['lead_days'])!=int(row['lead_days']):raise RuntimeError('wrong lead in file')
    a=np.column_stack([z['branch1_lon'],z['branch1_lat']]).astype(float)
    b=np.column_stack([z['branch2_lon'],z['branch2_lat']]).astype(float)
 return a,b
# 合成几何单元测试
_ref=np.column_stack([np.linspace(121,144.6,300),np.linspace(22,35,300)])
_p,n=east_prefix(_ref);assert abs(_p[-1,0]-c.TERMINAL_LON)<1e-6
_b,_=after_lon(_p,122);assert abs(_b[0,0]-122)<1e-6
assert abs(poly_score(_b,_b)['rmse_km'])<1e-8
assert section_intersections(np.column_stack([np.linspace(121,126,25),np.linspace(23.5,23,25)]),SECTIONS['A'])
print('GEOMETRY PREFLIGHT PASS')

In [ ]:
# 7 ADT 正式配对：主表仅用完整已验收ADT流轴，绝不混入2023部分轨迹。
metric_file=TABLES/'03_真实预测与ADT逐起报误差.csv'
section_file=TABLES/'04_六断面逐起报交点误差.csv'
miss_file=TABLES/'05_ADT配对覆盖与拒绝原因.csv'
metric=[];sect=[];missing=[]
if not len(manifest):print('NO VERIFIED PREDICTIONS: 跳过真实预测分数，输出原因而非虚构数值。')
for _,rr in tqdm(manifest.iterrows(),total=len(manifest),desc='Forecast->ADT'):
    row=rr.to_dict();target=row['target_date'];ref=load_adt(target)
    if ref is None:
        missing.append({**{k:row[k] for k in ('qa_year','seed','issue_date','target_date','lead_days')},'reason':'no_complete_ADT_axis'})
        continue
    try:
        a,b=load_prediction(row)
        adt_upper,n_adt_e=east_prefix(ref)
        adt_main,n_adt_w=after_lon(adt_upper,float(lon2.min()))
        model_main,n_model_e=east_prefix(b)
        model_main,n_model_w=after_lon(model_main,float(lon2.min()))
        scores=poly_score(model_main,adt_main)
        metric.append({**{k:row[k] for k in ('model','qa_year','seed','issue_date','target_date','lead_days')},
             **{'main_'+k:v for k,v in scores.items()},'adt_crossings_terminal':n_adt_e,
             'pred_crossings_terminal':n_model_e,'adt_crossings_west':n_adt_w,
             'pred_crossings_west':n_model_w,'reference':'ADT full axis','development_year':True})
    except Exception as exc:
        missing.append({**{k:row[k] for k in ('qa_year','seed','issue_date','target_date','lead_days')},
                        'reason':'main_axis_'+str(exc)[:180]})
    # branch1独立评分，不要求另一分支能形成连续连接
    try:
        r1,_=latitude_segment(ref,float(lat1.min()),float(lat1.max()))
        sc=poly_score(a,r1)
        if metric and metric[-1]['issue_date']==row['issue_date'] and metric[-1]['seed']==row['seed'] and metric[-1]['lead_days']==row['lead_days']:
            metric[-1].update({'upstream_'+k:v for k,v in sc.items()})
    except Exception:pass
    # 断面：A优先B1；B/PN/C/D/E优先B2；只允许独特单交点入主表
    for key,line in SECTIONS.items():
        choices=[('Branch1',a),('Branch2',b)] if key=='A' else [('Branch2',b),('Branch1',a)]
        ar=section_intersections(ref,line)
        options=[(source,section_intersections(x,line)) for source,x in choices]
        chosen=next(((source,q) for source,q in options if len(q)==1),None)
        n_pred=sum(len(q) for _,q in options)
        rec={k:row[k] for k in ('qa_year','seed','issue_date','target_date','lead_days')}
        rec.update({'section':key,'adt_n_intersections':len(ar),'pred_n_intersections':n_pred,
                    'pred_branch':chosen[0] if chosen else 'none','reference':'ADT full axis'})
        if len(ar)==1 and chosen is not None and n_pred==1:
            obs=ar[0];forecast=chosen[1][0]
            rec.update({'adt_distance_km':obs['section_distance_km'],
                        'forecast_distance_km':forecast['section_distance_km'],
                        'signed_error_km':forecast['section_distance_km']-obs['section_distance_km'],
                        'abs_error_km':abs(forecast['section_distance_km']-obs['section_distance_km']),
                        'paired_valid':True})
        else:
            rec.update({'paired_valid':False,'reason':'missing_or_multiple_intersections'})
        sect.append(rec)

metric_df=pd.DataFrame(metric);sec_df=pd.DataFrame(sect);missing_df=pd.DataFrame(missing)
for df,path in [(metric_df,metric_file),(sec_df,section_file),(missing_df,miss_file)]:
    df.to_csv(path,index=False,encoding='utf-8-sig')
print('ADT pairs',len(metric_df),'of',len(manifest),'all failures/ADT missing',len(missing_df))
if len(metric_df):display(metric_df.groupby(['qa_year','lead_days']).agg(n=('main_rmse_km','count'),rmse=('main_rmse_km','mean'),mae=('main_mae_km','mean')).round(2))

In [ ]:
# 8 年份/提前期统计；成对日期分块 CI；六断面位置均值和覆盖率。
def date_block_ci(frame,col='main_rmse_km',n=400,block_days=30,seed=20261016):
    # 先按 target_date 聚合重复起报和seed，然后在有序日期上进行块抽样。
    x=frame.groupby('target_date',as_index=False)[col].mean().sort_values('target_date')[col].dropna().to_numpy(float)
    if len(x)<10:return [np.nan,np.nan]
    rng=np.random.default_rng(seed);N=len(x);block=min(N,block_days);out=[]
    for _ in range(n):
        ids=[]
        while len(ids)<N:
            start=int(rng.integers(N));ids.extend((start+np.arange(block))%N)
        out.append(float(x[np.asarray(ids[:N],int)].mean()))
    return np.percentile(out,[2.5,97.5]).astype(float).tolist()
summary=[]
if len(metric_df):
    for (year,lead),g in metric_df.groupby(['qa_year','lead_days']):
        ci=date_block_ci(g,n=c.BOOT_REPS)
        n_forecast=len(manifest[(manifest.qa_year==year)&(manifest.lead_days==lead)])
        summary.append({'qa_year':year,'lead_days':lead,'n_forecast':n_forecast,'n_matched':len(g),
            'coverage_frac':len(g)/max(1,n_forecast),'n_unique_target_dates':g.target_date.nunique(),
            'mean_daily_rmse_km':g.main_rmse_km.mean(),'median_daily_rmse_km':g.main_rmse_km.median(),
            'mean_daily_mae_km':g.main_mae_km.mean(),'RMSE_block_CI95_lower':ci[0],
            'RMSE_block_CI95_upper':ci[1],
            'n_multicross_adt':int((g.adt_crossings_terminal>1).sum())})
S=pd.DataFrame(summary)
S.to_csv(TABLES/'06_论文主表_预测vsADT独立参考.csv',index=False,encoding='utf-8-sig')
sec_summary=[]
if len(sec_df):
    for (year,lead,section),g in sec_df.groupby(['qa_year','lead_days','section']):
        q=g[g.paired_valid==True].copy()
        sec_summary.append({'qa_year':year,'lead_days':lead,'section':section,
            'n_reference_days':g.target_date.nunique(),'n_valid_pairs':len(q),'coverage_frac':len(q)/max(1,len(g)),
            'mean_bias_km':q.signed_error_km.mean() if len(q) else np.nan,
            'rmse_km':float(np.sqrt(np.mean(q.signed_error_km**2))) if len(q) else np.nan,
            'mae_km':q.abs_error_km.mean() if len(q) else np.nan,
            'forecast_vs_ADT_r':q[['adt_distance_km','forecast_distance_km']].corr().iloc[0,1] if len(q)>2 else np.nan})
T=pd.DataFrame(sec_summary);T.to_csv(TABLES/'07_论文六断面_预测vsADT.csv',index=False,encoding='utf-8-sig')
coverage=[]
for (year,lead),g in manifest.groupby(['qa_year','lead_days']):
    matched=metric_df[(metric_df.qa_year==year)&(metric_df.lead_days==lead)] if len(metric_df) else pd.DataFrame()
    coverage.append({'qa_year':year,'lead_days':lead,'total_model_forecasts':len(g),
      'paired_complete_adt':len(matched),'missing_or_failed':len(g)-len(matched),
      'unique_issue_dates':g.issue_date.nunique(),'unique_paired_target_dates':matched.target_date.nunique() if len(matched) else 0})
pd.DataFrame(coverage).to_csv(TABLES/'08_覆盖与潜在非随机缺测.csv',index=False,encoding='utf-8-sig')
# 月统计只做2014和2019研发年份内变化，禁止假称31年趋势
if len(sec_df):
    mm=sec_df[sec_df.paired_valid==True].copy();mm['month']=pd.to_datetime(mm.target_date).dt.to_period('M').astype(str)
    mm.groupby(['qa_year','section','lead_days','month'],as_index=False).agg(
        observed_ADT_km=('adt_distance_km','mean'),forecast_km=('forecast_distance_km','mean'),
        signed_error_km=('signed_error_km','mean'),n=('signed_error_km','size'))        .to_csv(TABLES/'09_两年开发样本月尺度六断面变化_非长期趋势.csv',index=False,encoding='utf-8-sig')
print('Summary rows',len(S),'six-section rows',len(T))
if len(S):display(S.round(2))

In [ ]:
# 9 英文论文绘图，图例说明研发年份和真实完整ADT覆盖，绝不选择性遗漏不利日期。
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.20})
if len(S):
    fig,axs=plt.subplots(1,2,figsize=(11.6,4.0),sharey=True)
    for ax,year in zip(axs,c.QA_YEARS):
        g=S[S.qa_year==year].sort_values('lead_days')
        if not len(g):continue
        ax.errorbar(g.lead_days,g.mean_daily_rmse_km,yerr=[g.mean_daily_rmse_km-g.RMSE_block_CI95_lower,
            g.RMSE_block_CI95_upper-g.mean_daily_rmse_km],fmt='o-',capsize=4,linewidth=1.7,label='R65-CrossDrop60 vs ADT')
        ax.set(title=f'Development year {year}',xlabel='Lead time (days)',xticks=list(c.LEADS))
        ax.legend(loc='upper left')
    axs[0].set_ylabel('Daily common-domain RMSE (km)')
    fig.suptitle('Independent altimetry reference comparison (non-blind development years)',fontsize=11)
    fig.tight_layout();fig.savefig(FIG/'Fig01_ForecastADT_ErrorByLead.png',dpi=c.FIG_DPI);fig.savefig(FIG/'Fig01_ForecastADT_ErrorByLead.pdf');plt.close(fig)
if len(T):
    fig,axs=plt.subplots(1,2,figsize=(12,4),sharey=True)
    for ax,year in zip(axs,c.QA_YEARS):
        f=T[T.qa_year==year]
        for lead in c.LEADS:
            g=f[f.lead_days==lead].set_index('section').reindex(SECTIONS.keys())
            ax.plot(list(SECTIONS),g.rmse_km.values,marker='o',label=f'{lead} days')
        ax.set(title=f'{year} / valid unique section intersections only',xlabel='Section')
        ax.legend()
    axs[0].set_ylabel('Section-crossing RMSE (km)')
    fig.tight_layout();fig.savefig(FIG/'Fig02_SixSection_ADT_Forecast.png',dpi=c.FIG_DPI);fig.savefig(FIG/'Fig02_SixSection_ADT_Forecast.pdf');plt.close(fig)
if len(metric_df):
    fig,axs=plt.subplots(2,1,figsize=(12,6),sharex=False)
    for ax,year in zip(axs,c.QA_YEARS):
        t=metric_df[metric_df.qa_year==year]
        for lead in c.LEADS:
            v=t[t.lead_days==lead].groupby('target_date',as_index=False).main_rmse_km.mean()
            ax.plot(pd.to_datetime(v.target_date),v.main_rmse_km,label=f'{lead} days',alpha=.7,lw=.9)
        ax.set(title=f'{year} (all matched dates; ADT full-axis only)',ylabel='RMSE (km)')
        ax.legend(ncol=3)
    fig.tight_layout();fig.savefig(FIG/'Fig03_ActualTargetDates_Error.png',dpi=c.FIG_DPI);fig.savefig(FIG/'Fig03_ActualTargetDates_Error.pdf');plt.close(fig)
print('Generated figures:',len(list(FIG.glob('*.png'))))

In [ ]:
# 10 Finalize: strict audit, exact status, ZIP/SHA256 even on partial preflight failure.
status={
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'protocol':'R7.4 exact frozen R65-CrossDrop60 + complete ADT axis',
 'dataset_sha256':DATASET_SHA,
 'model':'R65-CrossDrop60 (single model, no dual-model switching)',
 'development_years':list(c.QA_YEARS),'seeds':list(c.SEEDS),'leads_days':list(c.LEADS),
 'n_checkpoints_expected':len(FOLDS)*len(c.SEEDS),'n_checkpoints_cid_pass':int(ckpt_audit.passed.sum()),
 'n_exported_forecasts':int(len(manifest)),'n_adt_paired_forecasts':int(len(metric_df)),
 'prediction_vs_adt_finished':bool(len(metric_df)>0 and ckpt_audit.passed.all()),
 'complete_all_requested_checkpoints':bool(ckpt_audit.passed.all()),
 'partial_2023_adt_used_in_main_evaluation':False,
 'independent_blind':False,
 'scope_limits':['Only QA development years 2014/2019 have known frozen weights',
    'GLORYS 50m and satellite ADT surface ridges are not identical physical quantities',
    'The two forecast branches are scored separately; no artificial connecting segment',
    'Results are subject to nonrandom ADT axis QC failures',
    'CI resamples serially correlated target dates and does not include reference-algorithm bias'],
}
(REP/'R74_冻结模型与独立验证完成状态.json').write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
(REP/'R74_论文科学说明.txt').write_text('R7.4 uses only the frozen R65-CrossDrop60 best.pt checkpoints. Original dataset, train-only EOF, causal QA issue dates and exact CID are recomputed. Forecast leads 30/60/90d are verified against satellite-ADT complete axes by exact target date. Results are from development years 2014/2019; not an unseen multi-year blind test. Missing ADT references are not treated as model error. Regional trend over 1993–2023 is handled separately by R70–R73; 2014/2019 predictions cannot establish a multi-decade forecast trend.\n',encoding='utf-8')
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for base in (TABLES,REP,FIG):
        for p in sorted(base.rglob('*')):
            if p.is_file():archive.write(p,arcname=str(p.relative_to(RUN)))
    for p in sorted(SNAP.glob('*.npz')):archive.write(p,arcname=str(p.relative_to(RUN)))
    files=sorted(PATHS.glob('R65-CrossDrop60_*_L*.npz'))
    if c.INCLUDE_ALL_PREDICTIONS_IN_ZIP:take=files
    else:take=files[::max(1,len(files)//max(1,c.ZIP_SAMPLE_PREDICTIONS))][:c.ZIP_SAMPLE_PREDICTIONS]
    for p in take:archive.write(p,arcname='example_forecast_paths/'+p.name)
with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()
    if bad:raise RuntimeError('ZIP CRC failed: '+str(bad))
sha=file_sha256(ZIP);Path(str(ZIP)+'.sha256').write_text(sha+'  '+ZIP.name+'\n',encoding='utf-8')
print('R74 status',json.dumps(status,ensure_ascii=False,indent=2))
print('ZIP:',ZIP,'bytes',ZIP.stat().st_size,'SHA256',sha)
display(FileLink(str(ZIP)))
if not status['complete_all_requested_checkpoints']:
    print('WARNING: checkpoint(s) absent or CID mismatch. Existing predictions must not be labeled comprehensive.')

### 方法及结果声明

- **空间分界**：主报告主流区 Branch2 (122.0°E→144.3°E) 与 ADT 同域比较；台湾东侧 Branch1 (固定纬线 23–25.5°N) 单独评分。两分支**不做直线拼接**。
- **统计方法**：每个预测起报 × 提前期逐日算球面最近路径距离，汇总时明确写为「逐日对称 RMSE 的算术平均」，不是合并所有空间点计算的整体 RMS。六断面仅单值真实交点列入主表，多交点与缺交点记录覆盖信息。
- **不确定性**：按目标日期先聚合重复起报/种子，再做日期连续块 Bootstrap；同一个年份内部的区间，不能解释为真正跨年份泛化 CI。
- **预报长期趋势**：既有 R65 已训练权重仅支持 2014/2019 研发年。要分析 1993–2023 年 31 年的预报长期趋势，必须重新执行严格的多年滚动后报，这属于审稿任务【2】，不能凭这两个年份推断。
- **文件复现**：真实预测轨迹本机保存在 `forecast_paths/*.npz`，源数据和权重 SHA256 记录在 Manifest 与报告中。默认 ZIP 只附少量可审计样例以便上传；需要完整预测轨迹勾选 `INCLUDE_ALL_PREDICTIONS_IN_ZIP=True`。